# Imports


In [101]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import networkx as nx
from scipy.stats import fisher_exact
from pyvis.network import Network
import numpy as np

sns.set_theme(style="whitegrid")


# Filter all the file to obtain just the chosen info (filtri scelti sul portale)


## Filter raws for mutation, sv e gene panel file


In [102]:
import pandas as pd
import os

def filter_samples_by_id(reference_filename, target_files_list, folder_path, output_prefix="F_"):
    """
    Mantiene nei target_files solo le righe il cui Sample_Id è presente nel reference_csv.
    """
    # Costruisco il path completo del file di riferimento
    reference_csv_path = os.path.join(folder_path, reference_filename)
    
    try:
        ref_df = pd.read_csv(reference_csv_path, sep='\t')
        valid_ids = set(ref_df['Sample_Id'].astype(str).unique())
        print(f"\n--- Analisi cartella: {folder_path} ---")
        print(f"Caricati {len(valid_ids)} ID univoci dal master.")
    except Exception as e:
        print(f"Errore caricamento master {reference_csv_path}: {e}")
        return

    for filename in target_files_list:
        file_input_path = os.path.join(folder_path, filename)
        
        if not os.path.exists(file_input_path):
            print(f"File non trovato: {filename}, salto...")
            continue
            
        print(f"Filtraggio in corso per: {filename}...")
        target_df = pd.read_csv(file_input_path, sep='\t')
        print(f"Righe originali in {filename}: {len(target_df)}")
        
        if 'Sample_Id' in target_df.columns:
            # Filtro righe
            filtered_df = target_df[target_df['Sample_Id'].astype(str).isin(valid_ids)]
            
            # Costruisco il path di output (F_nomefile.txt) dentro la cartella corretta
            output_name = f"{output_prefix}{filename}"
            output_full_path = os.path.join(folder_path, output_name)
            
            filtered_df.to_csv(output_full_path, index=False, sep='\t')
            print(f"Salvato: {output_name} ({len(filtered_df)} righe mantenute)")
        else:
            print(f"Colonna 'Sample_Id' non trovata in {filename}. File saltato.")

# --- CONFIGURAZIONE PATH E FILE ---

# 1. Lista dei nomi dei file (senza path) da filtrare in ogni cartella
file_da_filtrare = ["data_mutations.txt"] 

# 2. Esecuzione per le diverse coorti
# KRAS PANCREAS
path_pancreas = "./full_try/kras_pancreas/"
filter_samples_by_id('F_nokras_pancreas.csv', file_da_filtrare, path_pancreas)

# KRAS LUNG
path_lung = "./full_try/kras_lung/"
filter_samples_by_id('F_nokras_lung.csv', file_da_filtrare, path_lung)

# KRAS COLON
path_colon = "./full_try/kras_colon/"
filter_samples_by_id('F_nokras_colon.csv', file_da_filtrare, path_colon)

Errore caricamento master ./full_try/kras_pancreas/F_nokras_pancreas.csv: [Errno 2] No such file or directory: './full_try/kras_pancreas/F_nokras_pancreas.csv'
Errore caricamento master ./full_try/kras_lung/F_nokras_lung.csv: [Errno 2] No such file or directory: './full_try/kras_lung/F_nokras_lung.csv'
Errore caricamento master ./full_try/kras_colon/F_nokras_colon.csv: [Errno 2] No such file or directory: './full_try/kras_colon/F_nokras_colon.csv'


### Filter the columns of cna file


In [103]:
def filter_columns_by_master(path, master_path, target_path, output_path):

    if not os.path.exists(f"{path}/{master_path}"):
        print(f"File not found: {path}/{master_path}. Skipping this cohort.")
        return

    master_df = pd.read_csv(f"{path}/{master_path}", sep='\t')
    
    # Crea un set di Sample_Id validi
    valid_ids = set(master_df['Sample_Id'].astype(str).unique())
    print(f"ID univoci trovati nel master: {len(valid_ids)}")

    # 2. Carica il file target (quello da cui levare le colonne)
    target_df = pd.read_csv(f"{path}/{target_path}", sep='\t')
    print(f"Shape del target_df: {target_df.shape}")
    
    # 3. Identifica le colonne da tenere
    # Vogliamo tenere 'Hugo_Symbol' (o altre colonne descrittive iniziali) 
    # E tutte le colonne il cui nome è presente in valid_ids
    columns_to_keep = []
    columns_removed = 0
    
    for col in target_df.columns:
        if col == 'Hugo_Symbol' or col in valid_ids:
            columns_to_keep.append(col)
        else:
            columns_removed += 1
            
    # 4. Filtra il dataframe
    filtered_df = target_df[columns_to_keep]
    
    # 5. Salva il risultato
    filtered_df.to_csv(f"{path}/{output_path}", sep='\t', index=False)
    
    print(f"Colonne totali rimosse: {columns_removed}")
    print(f"Colonne rimaste: {len(columns_to_keep)}")
    print(f"File salvato in: {path}/{output_path}")



path = "./kras_pancreas"
filter_columns_by_master(path, 'F_nokras_pancreas.csv', 'data_cna.txt', 'F_data_cna.txt')

path = "./kras_lung"
filter_columns_by_master(path, 'F_nokras_lung.csv', 'data_cna.txt', 'F_data_cna.csv')

path = "./kras_colon"
filter_columns_by_master(path, 'F_nokras_colon.csv', 'data_cna.txt', 'F_data_cna.csv')

ID univoci trovati nel master: 1672
Shape del target_df: (702, 25035)
Colonne totali rimosse: 23365
Colonne rimaste: 1670
File salvato in: ./kras_pancreas/F_data_cna.txt
ID univoci trovati nel master: 3833
Shape del target_df: (702, 25035)
Colonne totali rimosse: 21201
Colonne rimaste: 3834
File salvato in: ./kras_lung/F_data_cna.csv
ID univoci trovati nel master: 2347
Shape del target_df: (702, 25035)
Colonne totali rimosse: 22688
Colonne rimaste: 2347
File salvato in: ./kras_colon/F_data_cna.csv


### Colonne sus


In [104]:
import pandas as pd
import numpy as np

# Carichiamo il file (usiamo low_memory=False solo per l'ispezione)
path_test = "./kras_pancreas/data_mutations.txt"
df = pd.read_csv(path_test, sep='\t', low_memory=False)

indices_incriminati = [44, 49, 88]

print("--- ANALISI COLONNE MIXED TYPES ---")

for idx in indices_incriminati:
    col_name = df.columns[idx]
    print(f"\nColonna {idx}: '{col_name}'")
    
    # Vediamo i tipi di dati reali presenti
    tipi_rilevati = df.iloc[:, idx].apply(lambda x: type(x)).unique()
    print(f"Tipi Python trovati: {tipi_rilevati}")
    
    # Mostriamo i primi 10 valori unici per capire il contenuto
    valori_unici = df.iloc[:, idx].unique()[:10]
    print(f"Esempio valori: {valori_unici}")
    
    # Controlliamo se ci sono stringhe che dovrebbero essere numeri
    n_stringhe = df.iloc[:, idx].apply(lambda x: isinstance(x, str)).sum()
    print(f"Numero di righe con testo (str): {n_stringhe}")

--- ANALISI COLONNE MIXED TYPES ---

Colonna 44: 'Exon_Number'
Tipi Python trovati: [<class 'float'> <class 'str'>]
Esempio valori: [nan '1-Jan' '3-Jan' '22/22' '9-Jan' 'Jan-33' '1/3/08' '4-Jan' '20-Jan'
 '16-Jan']
Numero di righe con testo (str): 16

Colonna 49: 'ALLELE_NUM'
Tipi Python trovati: [<class 'float'> <class 'str'>]
Esempio valori: [nan 'NEWRECORD']
Numero di righe con testo (str): 16236

Colonna 88: 'IS_NEW'
Tipi Python trovati: [<class 'float'> <class 'str'>]
Esempio valori: [nan 'NEWRECORD']
Numero di righe con testo (str): 12258


# Co-occurrence study


## Matrici


In [105]:
# --- FUNZIONI STANDARD ---

def generate_cooccurrence_data(filtered_mutations_file, cohort_name, output_dir):
    """
    Legge il file MAF filtrato per pazienti, applica filtri funzionali sulle varianti,
    genera matrici di co-occorrenza e visualizza heatmap.
    """
    print(f"\n=== Generazione Mappa Co-occorrenza: {cohort_name.upper()} ===")

    print(output_dir)

    # 1. Caricamento Dati (usiamo low_memory=False per i dtypes misti visti precedentemente)
    if not os.path.exists(filtered_mutations_file):
        print(f"File non trovato: {filtered_mutations_file}, salto coorte.")
        return None
        
    df = pd.read_csv(filtered_mutations_file, sep='\t', low_memory=False)

    # 2. FILTRO STANDARD DELLE VARIANTI (Health Lab Criterion)
    # Definiamo quali classificazioni di varianti consideriamo "funzionali" o "non-silenti"
    functional_variants = [
        'Missense_Mutation',
        'Nonsense_Mutation',
        'Frame_Shift_Del',
        'Frame_Shift_Ins',
        'In_Frame_Del',
        'In_Frame_Ins',
        'Splice_Site',
        'Translation_Start_Site',
        'Nonstop_Mutation'
        # Escludiamo: Silent, Intron, 3'UTR, 5'UTR, 5'Flank, etc.
    ]

    # Applichiamo il filtro
    df_func = df[df['Variant_Classification'].isin(functional_variants)].copy()
    
    n_original = df.shape[0]
    n_filtered = df_func.shape[0]
    print(f"Filtro Varianti: Mantenute {n_filtered}/{n_original} mutazioni funzionali.")

    if df_func.empty:
        print("Nessuna mutazione funzionale trovata. Salto coorte.")
        return None

    # 3. CREAZIONE MATRICE BINARIA PAZIENTE-GENE
    # Usiamo crosstab per contare le mutazioni per gene per paziente.
    # crosstab gestisce automaticamente i duplicati (paziente con >1 mutazione nello stesso gene).
    pat_gene_counts = pd.crosstab(df_func['Sample_Id'], df_func['Hugo_Symbol'])

    # Trasformiamo in binario (1 se >= 1 mutazione, 0 altrimenti)
    binary_matrix = (pat_gene_counts > 0).astype(int)
    
    n_patients = binary_matrix.shape[0]
    n_genes = binary_matrix.shape[1]
    print(f"Matrice Binaria creata: {n_patients} pazienti x {n_genes} geni.")

    # Salvataggio Matrice Binaria (utile per input esterni)
    binary_path = os.path.join(output_dir, f"M_binary_pat_gene_{cohort_name}.csv")
    binary_matrix.to_csv(binary_path, sep='\t')
    print(f"Salvata matrice binaria: {binary_path}")

    # 4. CALCOLO MATRICE CO-OCCORRENZA GENE-GENE
    # Formula matematica efficiente: Prodotto scalare della matrice trasposta per se stessa (T * Matrix)
    # I valori diagonali conterranno il totale delle mutazioni per gene.
    # I valori off-diagonal conterranno il numero di pazienti con entrambi i geni mutati.
    co_occ_matrix = binary_matrix.T.dot(binary_matrix)

    # Salvataggio Matrice Co-occorrenza completa
    co_occ_path = os.path.join(output_dir, f"M_cooccurrence_gene_gene_{cohort_name}.tsv")
    co_occ_matrix.to_csv(co_occ_path, sep='\t')
    print(f"Salvata matrice co-occorrenza gene-gene: {co_occ_path}")

    # 5. FOCUS SU RAS (KRAS)
    # Identifichiamo i top partner di KRAS (assumendo sia il gene target principale)
    target_gene = 'KRAS'
    if target_gene in co_occ_matrix.columns:
        kras_partners = co_occ_matrix[target_gene].sort_values(ascending=False)
        # Rimuoviamo l'auto-occorrenza (KRAS con KRAS)
        kras_partners = kras_partners.drop(target_gene)
        
        # Calcolo frequenza target
        kras_mutated_pats = binary_matrix[target_gene].sum()
        perc_kras = (kras_mutated_pats / n_patients) * 100

        print(f"\n--- Top 10 Geni co-mutati con {target_gene} ---")
        print(f"({target_gene} mutato in {kras_mutated_pats}/{n_patients} pazienti - {perc_kras:.1f}%)")
        print(kras_partners.head(10))
    else:
        print(f"\nGene target {target_gene} non trovato nella matrice.")

    # Identifichiamo i Top 20 geni più mutati in assoluto per la visualizzazione
    gene_freqs = binary_matrix.sum().sort_values(ascending=False)
    top_genes_for_plot = gene_freqs.index[:20]

    return co_occ_matrix, top_genes_for_plot, n_patients

def visualize_cooccurrence_heatmap(co_occ_matrix, top_genes, n_patients, cohort_name, output_dir):
    """Genera e salva un heatmap della co-occorrenza per i geni selezionati."""
    print(f"\nGenerazione Heatmap per {cohort_name}...")

    # Selezioniamo il subset dei dati
    subset_matrix = co_occ_matrix.loc[top_genes, top_genes]

    # Creiamo una maschera per nascondere la parte superiore (duplicata e diagonale) per pulizia visiva
    mask = np.triu(np.ones_like(subset_matrix, dtype=bool))

    # Configurazione plot
    plt.figure(figsize=(14, 12))
    sns.set_theme(style="white")

    # Generazione Heatmap
    heatmap = sns.heatmap(subset_matrix,
                        mask=mask,
                        cmap="YlOrRd", # Scala colori Giallo-Arancio-Rosso (frequenza alta)
                        annot=True, # Mostra i numeri
                        fmt='d', # Formato interi
                        linewidths=.5,
                        cbar_kws={'label': 'Numero Pazienti Co-mutati'})

    plt.title(f"Mappa di Co-occorrenza Top Geni: {cohort_name.upper()}\n(N total pazienti funzionali = {n_patients})", fontsize=16)
    plt.ylabel("") # Rimuoviamo label assi ridondanti
    plt.xlabel("")

    # Salvataggio immagine
    plot_path = os.path.join(output_dir, f"Heatmap_cooccurrence_{cohort_name}.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Heatmap salvata: {plot_path}")


# === ESECUZIONE PIPELINE ===

# Definiamo i file input (i file F_ generati nello step precedente) e i nomi coorte
studies_to_map = [
    ("./kras_pancreas/F_data_mutations.txt", "pancreas"),
    ("./kras_lung/F_data_mutations.txt", "lung"),
    ("./kras_colon/F_data_mutations.txt", "colon")
]

for file_input, name in studies_to_map:
    # Usiamo la cartella del file input come cartella output
    folder_path = os.path.dirname(file_input) + "/co_occurr_output"
    
    # Eseguiamo il calcolo dei dati
    result = generate_cooccurrence_data(file_input, name, folder_path)
    
    # Se i dati sono stati generati correttamente, creiamo l'heatmap
    if result:
        co_occ_mat, top_genes_list, n_pats = result
        visualize_cooccurrence_heatmap(co_occ_mat, top_genes_list, n_pats, name, folder_path)

print("\n=== Phase 1 - Step 2 Completata: Dati pronti per la Network Analysis avanzata ===")


=== Generazione Mappa Co-occorrenza: PANCREAS ===
./kras_pancreas/co_occurr_output
Filtro Varianti: Mantenute 6140/6150 mutazioni funzionali.
Matrice Binaria creata: 1400 pazienti x 446 geni.
Salvata matrice binaria: ./kras_pancreas/co_occurr_output/M_binary_pat_gene_pancreas.csv
Salvata matrice co-occorrenza gene-gene: ./kras_pancreas/co_occurr_output/M_cooccurrence_gene_gene_pancreas.tsv

--- Top 10 Geni co-mutati con KRAS ---
(KRAS mutato in 1400/1400 pazienti - 100.0%)
Hugo_Symbol
TP53      1102
CDKN2A     344
SMAD4      314
ARID1A     109
RNF43       94
KDM6A       64
TGFBR2      63
KMT2D       51
ATM         44
RBM10       43
Name: KRAS, dtype: int64

Generazione Heatmap per pancreas...
Heatmap salvata: ./kras_pancreas/co_occurr_output/Heatmap_cooccurrence_pancreas.png

=== Generazione Mappa Co-occorrenza: LUNG ===
./kras_lung/co_occurr_output
Filtro Varianti: Mantenute 11715/11792 mutazioni funzionali.
Matrice Binaria creata: 1233 pazienti x 502 geni.
Salvata matrice binaria: .

## Filtro statistico Del Re


In [106]:
def analyze_significance(binary_matrix_file, target_gene='KRAS', p_threshold=0.05, log2or_threshold=3):
    """
    Calcola Fisher Exact Test e Log2OR per KRAS contro tutti gli altri geni.
    """
    # Carica la matrice binaria (Pazienti x Geni)
    df_bin = pd.read_csv(binary_matrix_file, sep='\t', index_col=0)
    
    if target_gene not in df_bin.columns:
        print(f"Errore: {target_gene} non trovato nella matrice.")
        return None

    results = []
    genes = [g for g in df_bin.columns if g != target_gene]
    
    n_total = len(df_bin)
    print(f"Analisi di {len(genes)} geni su {n_total} pazienti...")

    for gene_x in genes:
        # Costruzione Tabella di Contingenza
        # a: Entrambi mutati
        # b: Solo Target mutato
        # c: Solo Gene X mutato
        # d: Entrambi Wild-Type
        a = ((df_bin[target_gene] == 1) & (df_bin[gene_x] == 1)).sum()
        b = ((df_bin[target_gene] == 1) & (df_bin[gene_x] == 0)).sum()
        c = ((df_bin[target_gene] == 0) & (df_bin[gene_x] == 1)).sum()
        d = ((df_bin[target_gene] == 0) & (df_bin[gene_x] == 0)).sum()

        table = [[a, b], [c, d]]
        
        # Fisher Exact Test
        odds_ratio, p_value = fisher_exact(table, alternative='greater')
        
        # Calcolo Log2OR (gestendo divisioni per zero o OR=0)
        if odds_ratio > 0:
            log2or = np.log2(odds_ratio)
        else:
            log2or = -np.inf # Associazione negativa (Mutual Exclusion)

        results.append({
            'Gene_B': gene_x,
            'Co_Occurrence_Count': a,
            'P_Value': p_value,
            'Odds_Ratio': odds_ratio,
            'Log2OR': log2or
        })

    results_df = pd.DataFrame(results)

    # APPLICAZIONE FILTRI (Health Lab standard)
    significant_df = results_df[
        (results_df['P_Value'] < p_threshold) & 
        (results_df['Log2OR'] > log2or_threshold)
    ].sort_values(by='P_Value')

    return significant_df

# Pancreas
binary_file = "./full_try/kras_pancreas/co_occurr_output/M_binary_pat_gene_pancreas.csv"
if os.path.exists(binary_file):
    df_sig = analyze_significance(binary_file, target_gene='KRAS')
    
    # Salva i risultati per la Rete
    output_path = "./full_try/kras_pancreas/co_occurr_output/Significant_Cooccurrence_KRAS.csv"
    df_sig.to_csv(output_path, sep='\t', index=False)
    
    print(f"\nAssociazioni trovate con P < 0.05 e Log2OR > 3: {len(df_sig)}")
    print(df_sig.head(10))
    
# Lung
binary_file = "./full_try/kras_lung/co_occurr_output/M_binary_pat_gene_lung.csv"
if os.path.exists(binary_file):
    df_sig = analyze_significance(binary_file, target_gene='KRAS')
    
    # Salva i risultati per la Rete
    output_path = "./kras_lung/co_occurr_output/Significant_Cooccurrence_KRAS.csv"
    df_sig.to_csv(output_path, sep='\t', index=False)
    
    print(f"\nAssociazioni trovate con P < 0.05 e Log2OR > 3: {len(df_sig)}")
    print(df_sig.head(10))
    
# Colon
binary_file = "./kras_colon/co_occurr_output/M_binary_pat_gene_colon.csv"
if os.path.exists(binary_file):
    df_sig = analyze_significance(binary_file, target_gene='KRAS')
    
    # Salva i risultati per la Rete
    output_path = "./kras_colon/co_occurr_output/Significant_Cooccurrence_KRAS.csv"
    df_sig.to_csv(output_path, sep='\t', index=False)
    
    print(f"\nAssociazioni trovate con P < 0.05 e Log2OR > 3: {len(df_sig)}")
    print(df_sig.head(10))

Analisi di 501 geni su 1007 pazienti...

Associazioni trovate con P < 0.05 e Log2OR > 3: 0
Empty DataFrame
Columns: [Gene_B, Co_Occurrence_Count, P_Value, Odds_Ratio, Log2OR]
Index: []


## Networks (static visualization)


### Star topology (arcs connect only KRAS WITH other genes)


In [107]:
def build_ras_network(co_occ_file, min_cooccurrence, target_gene='KRAS', output_dir="."):
    """
    Costruisce la rete di co-occorrenza centrata su RAS e calcola le metriche.
    """
    if not os.path.exists(co_occ_file):
        print(f"   [ERRORE] File non trovato: {co_occ_file}")
        return None
    
    # 1. Caricamento della matrice
    co_occ_matrix = pd.read_csv(co_occ_file, sep='\t', index_col=0)
    
    if target_gene not in co_occ_matrix.columns:
        print(f"   [SKIP] {target_gene} non presente in questa coorte.")
        return None

    # 2. Creazione Grafo
    G = nx.Graph()
    partners = co_occ_matrix[target_gene]
    # Filtriamo i partner basandoci sulla soglia scelta
    significant_partners = partners[partners >= min_cooccurrence].drop(target_gene, errors='ignore')

    if significant_partners.empty:
        print(f"   [WARNING] Nessun gene soddisfa la soglia di co-occorrenza >= {min_cooccurrence}")
        return None

    # Aggiunta nodo centrale e partner
    G.add_node(target_gene, size=co_occ_matrix.loc[target_gene, target_gene])
    for gene, weight in significant_partners.items():
        G.add_node(gene, size=co_occ_matrix.loc[gene, gene])
        G.add_edge(target_gene, gene, weight=weight)

    print(f"   -> Rete creata: {G.number_of_nodes()} nodi (Hub principale: {target_gene})")
    print(f"   -> Soglia utilizzata: {min_cooccurrence} pazienti")

    # 3. Calcolo Metriche (Centralità)
    degree_cent = nx.degree_centrality(G)
    betweenness_cent = nx.betweenness_centrality(G)

    metrics = pd.DataFrame({
        'Gene': list(degree_cent.keys()),
        'Degree_Centrality': list(degree_cent.values()),
        'Betweenness_Centrality': list(betweenness_cent.values())
    }).sort_values(by='Degree_Centrality', ascending=False)

    # 4. Visualizzazione
    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G, k=0.5, seed=42) # Seed fisso per avere grafici riproducibili
    
    # Dimensione nodi proporzionale alle mutazioni totali
    node_sizes = [G.nodes[n].get('size', 1) * 10 for n in G.nodes]
    
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='skyblue', alpha=0.7)
    nx.draw_networkx_edges(G, pos, edge_color='gray', alpha=0.3)
    nx.draw_networkx_labels(G, pos, font_size=8)
    
    plt.title(f"Network: {target_gene} (Min Occ: {min_cooccurrence})")
    plt.axis('off')
    
    # Salvataggio: usa os.path.join per sicurezza sui diversi sistemi operativi
    output_img = os.path.join(output_dir, f"star_network_{min_cooccurrence}.png")
    plt.savefig(output_img, dpi=300, bbox_inches='tight')
    plt.close()
    
    return metrics

# --- ESECUZIONE ---

min_occ = 5

print("="*50)
print("PHASE 1: NETWORK ANALYSIS - GENE CO-OCCURRENCE")
print("="*50)

# 1. COLON
print(f"\n[1/3] ANALISI COORTE: COLON")
# FIX: Passiamo gli argomenti nell'ordine corretto
metrics_colon = build_ras_network(
    "./kras_colon/co_occurr_output/M_cooccurrence_gene_gene_colon.tsv", 
    min_occ, 
    target_gene='KRAS', 
    output_dir="./kras_colon/networks/"
)
if metrics_colon is not None:
    print("\n   Top 5 Hubs rilevati:")
    print(metrics_colon.head(5).to_string(index=False))

# 2. LUNG
print(f"\n[2/3] ANALISI COORTE: LUNG")
metrics_lung = build_ras_network(
    "./kras_lung/co_occurr_output/M_cooccurrence_gene_gene_lung.tsv", 
    min_occ, 
    target_gene='KRAS', 
    output_dir="./kras_lung/networks/"
)
if metrics_lung is not None:
    print("\n   Top 5 Hubs rilevati:")
    print(metrics_lung.head(5).to_string(index=False))

# 3. PANCREAS
print(f"\n[3/3] ANALISI COORTE: PANCREAS")
metrics_pancreas = build_ras_network(
    "./kras_pancreas/co_occurr_output/M_cooccurrence_gene_gene_pancreas.tsv", 
    min_occ, 
    target_gene='KRAS', 
    output_dir="./kras_pancreas/networks/"
)
if metrics_pancreas is not None:
    print("\n   Top 5 Hubs rilevati:")
    print(metrics_pancreas.head(5).to_string(index=False))

print("\n" + "="*50)
print("FINE ELABORAZIONE - Immagini salvate nelle cartelle")
print("="*50)

PHASE 1: NETWORK ANALYSIS - GENE CO-OCCURRENCE

[1/3] ANALISI COORTE: COLON
   -> Rete creata: 447 nodi (Hub principale: KRAS)
   -> Soglia utilizzata: 5 pazienti

   Top 5 Hubs rilevati:
  Gene  Degree_Centrality  Betweenness_Centrality
  KRAS           1.000000                     1.0
PDGFRA           0.002242                     0.0
PIK3R2           0.002242                     0.0
PIK3R1           0.002242                     0.0
PIK3CG           0.002242                     0.0

[2/3] ANALISI COORTE: LUNG
   -> Rete creata: 392 nodi (Hub principale: KRAS)
   -> Soglia utilizzata: 5 pazienti

   Top 5 Hubs rilevati:
   Gene  Degree_Centrality  Betweenness_Centrality
   KRAS           1.000000                     1.0
 PIK3CD           0.002558                     0.0
 PIK3CA           0.002558                     0.0
 PIK3C3           0.002558                     0.0
PIK3C2G           0.002558                     0.0

[3/3] ANALISI COORTE: PANCREAS
   -> Rete creata: 190 nodi (Hub p

### Full network (arcs connect KRAS partners too)


In [108]:
def build_full_ras_network(co_occ_file, min_cooccurrence, target_gene='KRAS', output_dir="."):
    """
    Costruisce una FULL Network: include archi tra tutti i geni che superano 
    la soglia di co-occorrenza, permettendo l'identificazione di cluster.
    """
    if not os.path.exists(co_occ_file):
        print(f"   [ERRORE] File non trovato: {co_occ_file}")
        return None
    
    # 1. Caricamento della matrice di co-occorrenza
    co_occ_matrix = pd.read_csv(co_occ_file, sep='\t', index_col=0)
    
    if target_gene not in co_occ_matrix.columns:
        print(f"   [SKIP] {target_gene} non presente in questa coorte.")
        return None

    # 2. Selezione dei geni rilevanti
    # Prendiamo il target_gene + tutti i geni che co-mutano con lui sopra la soglia
    partners = co_occ_matrix[target_gene]
    relevant_genes = partners[partners >= min_cooccurrence].index.tolist()
    
    if len(relevant_genes) < 2:
        print(f"   [WARNING] Troppi pochi geni (n={len(relevant_genes)}) sopra la soglia {min_cooccurrence}")
        return None

    # 3. Creazione del Grafo Full
    G = nx.Graph()
    
    # Estraiamo la sottomatrice con i soli geni rilevanti
    sub_matrix = co_occ_matrix.loc[relevant_genes, relevant_genes]
    
    # Aggiungiamo nodi e TUTTI gli archi che superano la soglia tra di loro
    for i, gene_a in enumerate(relevant_genes):
        # Dimensione del nodo basata sulla frequenza totale di mutazione (diagonale della matrice)
        G.add_node(gene_a, size=co_occ_matrix.loc[gene_a, gene_a])
        
        for j, gene_b in enumerate(relevant_genes):
            if i < j: # Evitiamo duplicati e l'auto-anello sulla diagonale
                weight = sub_matrix.loc[gene_a, gene_b]
                if weight >= min_cooccurrence:
                    G.add_edge(gene_a, gene_b, weight=weight)

    print(f"   -> Full Network creata: {G.number_of_nodes()} nodi e {G.number_of_edges()} archi.")

    # 4. Calcolo Metriche (ora molto più significative in una full network)
    degree_cent = nx.degree_centrality(G)
    betweenness_cent = nx.betweenness_centrality(G, weight='weight')
    
    metrics = pd.DataFrame({
        'Gene': list(degree_cent.keys()),
        'Degree_Centrality': list(degree_cent.values()),
        'Betweenness_Centrality': list(betweenness_cent.values()),
        'Total_Mutations': [G.nodes[n]['size'] for n in G.nodes]
    }).sort_values(by='Degree_Centrality', ascending=False)

    # 5. Visualizzazione
    plt.figure(figsize=(12, 10))
    # Il layout spring_layout simula forze fisiche: i nodi più connessi staranno vicini
    pos = nx.spring_layout(G, k=0.8, iterations=100, seed=42)
    
    # Nodi: colore diverso per il target_gene
    node_colors = ['orange' if n == target_gene else 'skyblue' for n in G.nodes]
    node_sizes = [np.sqrt(G.nodes[n]['size']) * 100 for n in G.nodes] # Radice quadrata per scalare meglio
    
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.8)
    
    # Archi: spessore basato sul peso della co-occorrenza
    weights = [G.edges[e]['weight'] for e in G.edges]
    # Normalizziamo gli spessori per la visualizzazione
    max_w = max(weights) if weights else 1
    edge_widths = [(w / max_w) * 5 for w in weights]
    
    nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='gray', alpha=0.4)
    nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')
    
    plt.title(f"Full Co-occurrence Network: {target_gene} & Partners\n(Soglia min. co-occorrenza: {min_cooccurrence})", fontsize=15)
    plt.axis('off')
    
    # Salvataggio
    output_img = os.path.join(output_dir, f"full_network_occ_{min_cooccurrence}.png")
    plt.savefig(output_img, dpi=300, bbox_inches='tight')
    plt.close()
    
    return metrics

# --- ESECUZIONE ---

min_occ = 5

print("="*60)
print("PHASE 1: FULL NETWORK ANALYSIS - IDENTIFYING MODULES")
print("="*60)

# Elenco delle coorti da processare
cohorts = [
    ("COLON", "./kras_colon/co_occurr_output/M_cooccurrence_gene_gene_colon.tsv", "./kras_colon/networks/"),
    ("LUNG", "./kras_lung/co_occurr_output/M_cooccurrence_gene_gene_lung.tsv", "./kras_lung/networks/"),
    ("PANCREAS", "./kras_pancreas/co_occurr_output/M_cooccurrence_gene_gene_pancreas.tsv", "./kras_pancreas/networks/")
]

for label, file_path, out_path in cohorts:
    print(f"\n[+] ANALISI COORTE: {label}")
    res_metrics = build_full_ras_network(file_path, min_occ, target_gene='KRAS', output_dir=out_path)
    
    if res_metrics is not None:
        print(f"\n   Top Hubs per {label} (ordinati per Degree Centrality):")
        # Mostriamo le colonne più importanti
        print(res_metrics[['Gene', 'Degree_Centrality', 'Betweenness_Centrality', 'Total_Mutations']].head(10).to_string(index=False))
        print("-" * 30)

print("\n" + "="*60)
print("FINE: Le Full Networks sono state salvate nelle rispettive cartelle.")
print("="*60)

PHASE 1: FULL NETWORK ANALYSIS - IDENTIFYING MODULES

[+] ANALISI COORTE: COLON
   -> Full Network creata: 447 nodi e 30713 archi.

   Top Hubs per COLON (ordinati per Degree Centrality):
   Gene  Degree_Centrality  Betweenness_Centrality  Total_Mutations
   KRAS           1.000000                0.053310             1007
    APC           0.943946                0.063708              805
  ZFHX3           0.847534                0.009266               87
  KMT2D           0.840807                0.012759               97
 PIK3CA           0.816143                0.011119              274
 ARID1A           0.816143                0.007006              128
   FAT1           0.789238                0.011273               78
  KMT2B           0.786996                0.005666               91
ANKRD11           0.782511                0.006960               66
  KMT2C           0.780269                0.009638               81
------------------------------

[+] ANALISI COORTE: LUNG
   -> F

## Networks (dinamic visualization)


### Star topology (arcs connect only KRAS WITH other genes)


In [109]:
import pandas as pd
import networkx as nx
import os
import numpy as np
from pyvis.network import Network

def build_interactive_star_network(co_occ_file, min_cooccurrence, target_gene='KRAS', output_dir="."):
    """
    Costruisce una STAR Network interattiva: solo KRAS al centro collegato ai suoi partner.
    """
    if not os.path.exists(co_occ_file):
        print(f"   [ERRORE] File non trovato: {co_occ_file}")
        return None
    
    # 1. Caricamento dati
    co_occ_matrix = pd.read_csv(co_occ_file, sep='\t', index_col=0)
    
    if target_gene not in co_occ_matrix.columns:
        print(f"   [SKIP] {target_gene} non presente.")
        return None

    # 2. Identificazione Partner diretti sopra soglia
    partners = co_occ_matrix[target_gene]
    significant_partners = partners[partners >= min_cooccurrence].drop(target_gene, errors='ignore')
    
    if significant_partners.empty:
        print(f"   [WARNING] Nessun partner per {target_gene} sopra soglia {min_cooccurrence}")
        return None

    # 3. Creazione Rete Pyvis
    # notebook=False è necessario per salvare il file HTML correttamente
    net = Network(height="750px", width="100%", bgcolor="white", font_color="black", notebook=False, cdn_resources='remote')    
    
    # Configurazione fisica: aumentiamo la repulsione per distanziare bene i partner dal centro
    net.barnes_hut(gravity=-10000, central_gravity=0.1, spring_length=250)

    # 4. Aggiunta del Nodo Centrale (KRAS)
    kras_total_muts = int(co_occ_matrix.loc[target_gene, target_gene])
    net.add_node(target_gene, 
                 label=target_gene, 
                 title=f"GENE TARGET: {target_gene}<br>Mutazioni Totali: {kras_total_muts}", 
                 size=np.sqrt(kras_total_muts) * 3, 
                 color="#e74c3c", # Rosso intenso per il target
                 borderWidth=4)

    # 5. Aggiunta dei Partner (solo collegati al centro)
    for gene, count in significant_partners.items():
        gene_total_muts = int(co_occ_matrix.loc[gene, gene])
        
        # Nodo partner
        net.add_node(gene, 
                     label=gene, 
                     title=f"Partner: {gene}<br>Mutazioni Totali: {gene_total_muts}<br>Co-occorrenza con {target_gene}: {count}", 
                     size=np.sqrt(gene_total_muts) * 2, 
                     color="#3498db") # Blu per i partner
        
        # Arco (solo dal centro al partner)
        net.add_edge(target_gene, gene, 
                     value=count, # Spessore basato sulla co-occorrenza
                     title=f"Pazienti con entrambi mutati: {count}",
                     color="#bdc3c7")

    # 6. Salvataggio
    output_html = os.path.join(output_dir, f"star_network_interattiva_min{min_cooccurrence}.html")
    net.save_graph(output_html)
    print(f"   -> Star Network interattiva salvata: {output_html}")
    
    return len(significant_partners)

# --- ESECUZIONE ---
min_occ = 5

print("="*60)
print("PHASE 1: GENERAZIONE STAR NETWORKS INTERATTIVE")
print("="*60)

cohorts = [
    ("COLON", "./kras_colon/co_occurr_output/M_cooccurrence_gene_gene_colon.tsv", "./kras_colon/networks/"),
    ("LUNG", "./kras_lung/co_occurr_output/M_cooccurrence_gene_gene_lung.tsv", "./kras_lung/networks/"),
    ("PANCREAS", "./kras_pancreas/co_occurr_output/M_cooccurrence_gene_gene_pancreas.tsv", "./kras_pancreas/networks/")
]

for label, file_path, out_path in cohorts:
    print(f"\n[+] Elaborazione coorte {label}...")
    n_partners = build_interactive_star_network(file_path, min_occ, output_dir=out_path)
    if n_partners:
        print(f"   Trovati {n_partners} partner significativi per KRAS.")

print("\n" + "="*60)
print("COMPLETATO: Apri i file .html per vedere KRAS al centro del suo ecosistema mutazionale.")
print("="*60)

PHASE 1: GENERAZIONE STAR NETWORKS INTERATTIVE

[+] Elaborazione coorte COLON...
   -> Star Network interattiva salvata: ./kras_colon/networks/star_network_interattiva_min5.html
   Trovati 446 partner significativi per KRAS.

[+] Elaborazione coorte LUNG...
   -> Star Network interattiva salvata: ./kras_lung/networks/star_network_interattiva_min5.html
   Trovati 391 partner significativi per KRAS.

[+] Elaborazione coorte PANCREAS...
   -> Star Network interattiva salvata: ./kras_pancreas/networks/star_network_interattiva_min5.html
   Trovati 189 partner significativi per KRAS.

COMPLETATO: Apri i file .html per vedere KRAS al centro del suo ecosistema mutazionale.


### Full network (arcs connect KRAS partners too)


In [110]:
def build_interactive_ras_network(co_occ_file, min_cooccurrence, target_gene='KRAS', output_dir="."):
    """
    Costruisce una rete interattiva HTML (Full Network) e calcola le metriche di centralità.
    """
    if not os.path.exists(co_occ_file):
        print(f"   [ERRORE] File non trovato: {co_occ_file}")
        return None
    
    # 1. Caricamento dati
    co_occ_matrix = pd.read_csv(co_occ_file, sep='\t', index_col=0)
    
    if target_gene not in co_occ_matrix.columns:
        print(f"   [SKIP] {target_gene} non presente.")
        return None

    # 2. Selezione Geni Significativi (Target + chi co-muta sopra soglia)
    partners = co_occ_matrix[target_gene]
    relevant_genes = partners[partners >= min_cooccurrence].index.tolist()
    
    if len(relevant_genes) < 2:
        print(f"   [WARNING] Solo {len(relevant_genes)} geni sopra soglia. Rete troppo piccola.")
        return None

    # 3. Creazione Grafo NetworkX (per calcolo metriche)
    G = nx.Graph()
    sub_matrix = co_occ_matrix.loc[relevant_genes, relevant_genes]
    
    for i, gene_a in enumerate(relevant_genes):
        total_muts = int(co_occ_matrix.loc[gene_a, gene_a])
        G.add_node(gene_a, size=total_muts, title=f"Gene: {gene_a}<br>Mutazioni Totali: {total_muts}")
        
        for j, gene_b in enumerate(relevant_genes):
            if i < j:
                weight = int(sub_matrix.loc[gene_a, gene_b])
                if weight >= min_cooccurrence:
                    G.add_edge(gene_a, gene_b, weight=weight, title=f"Co-occorrenza: {weight}")

    # 4. Calcolo Metriche
    degree_cent = nx.degree_centrality(G)
    betweenness_cent = nx.betweenness_centrality(G)
    metrics = pd.DataFrame({
        'Gene': list(degree_cent.keys()),
        'Degree_Centrality': list(degree_cent.values()),
        'Betweenness_Centrality': list(betweenness_cent.values())
    }).sort_values(by='Degree_Centrality', ascending=False)

    # 5. Trasformazione in Pyvis (Interattiva)
    # bgcolor="#222222" per tema dark, o "white" per tema light
    net = Network(height="750px", width="100%", bgcolor="white", font_color="black", notebook=False, cdn_resources='remote')   
     
    # Opzioni fisiche per distanziare i nodi automaticamente
    net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=200)

    for n in G.nodes:
        # Scaliamo la dimensione del nodo per la visibilità
        base_size = np.sqrt(G.nodes[n]['size']) * 2
        color = "#ff7f0e" if n == target_gene else "#1f77b4" # Arancio per KRAS, Blu per gli altri
        net.add_node(n, label=n, title=G.nodes[n]['title'], size=base_size, color=color)

    for e in G.edges:
        # Spessore arco basato sulla forza della co-occorrenza
        w = G.edges[e]['weight']
        net.add_edge(e[0], e[1], value=w, title=G.edges[e]['title'], color="#abb2b9")

    # Salvataggio HTML
    output_html = os.path.join(output_dir, f"full_network_interattiva_min{min_cooccurrence}.html")
    net.save_graph(output_html)
    print(f"   -> Rete interattiva salvata: {output_html}")
    
    return metrics

# --- ESECUZIONE ---
min_occ = 5

print("="*60)
print("PHASE 1: GENERAZIONE NETWORK INTERATTIVE (HTML)")
print("="*60)

configs = [
    ("COLON", "./kras_colon/co_occurr_output/M_cooccurrence_gene_gene_colon.tsv", "./kras_colon/networks/"),
    ("LUNG", "./kras_lung/co_occurr_output/M_cooccurrence_gene_gene_lung.tsv", "./kras_lung/networks/"),
    ("PANCREAS", "./kras_pancreas/co_occurr_output/M_cooccurrence_gene_gene_pancreas.tsv", "./kras_pancreas/networks/")
]

for label, file_path, out_path in configs:
    print(f"\n[+] Elaborazione {label}...")
    m = build_interactive_ras_network(file_path, min_occ, output_dir=out_path)
    if m is not None:
        print(f"   Top Hubs (Degree): {', '.join(m['Gene'].head(5).tolist())}")

print("\n" + "="*60)
print("FINE: Apri i file .html con il tuo browser per esplorare le reti.")
print("="*60)

PHASE 1: GENERAZIONE NETWORK INTERATTIVE (HTML)

[+] Elaborazione COLON...
   -> Rete interattiva salvata: ./kras_colon/networks/full_network_interattiva_min5.html
   Top Hubs (Degree): KRAS, APC, ZFHX3, KMT2D, PIK3CA

[+] Elaborazione LUNG...
   -> Rete interattiva salvata: ./kras_lung/networks/full_network_interattiva_min5.html
   Top Hubs (Degree): KRAS, TP53, KEAP1, STK11, PTPRD

[+] Elaborazione PANCREAS...
   -> Rete interattiva salvata: ./kras_pancreas/networks/full_network_interattiva_min5.html
   Top Hubs (Degree): KRAS, TP53, SMAD4, CDKN2A, RNF43

FINE: Apri i file .html con il tuo browser per esplorare le reti.
